In [1]:

from pyspark.sql import functions as F
from pyspark.sql.types import *

StatementMeta(, 54e69200-58a3-4797-94fd-192eee74d181, 3, Finished, Available, Finished, False)

In [2]:
df = spark.table("DE_Learning_LH.dbo.earth_quake")
display(df.limit(5))

StatementMeta(, 54e69200-58a3-4797-94fd-192eee74d181, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 891cc9d0-f171-4a44-ae32-5150f2d21962)

In [5]:

geometry_schema = StructType([
    StructField("type", StringType(), True),
    StructField("coordinates", ArrayType(DoubleType()), True)
])

properties_schema = StructType([
    StructField("mag", DoubleType(), True),
    StructField("place", StringType(), True),
    StructField("time", LongType(), True),
    StructField("updated", LongType(), True),
    StructField("tz", IntegerType(), True),   # nullable
    StructField("url", StringType(), True),
    StructField("detail", StringType(), True),
    StructField("felt", IntegerType(), True),
    StructField("cdi", DoubleType(), True),
    StructField("mmi", DoubleType(), True),
    StructField("alert", StringType(), True),
    StructField("status", StringType(), True),
    StructField("tsunami", IntegerType(), True),
    StructField("sig", IntegerType(), True),
    StructField("net", StringType(), True),
    StructField("code", StringType(), True),
    StructField("ids", StringType(), True),
    StructField("sources", StringType(), True),
    StructField("types", StringType(), True),
    StructField("nst", IntegerType(), True),
    StructField("dmin", DoubleType(), True),
    StructField("rms", DoubleType(), True),
    StructField("gap", IntegerType(), True),
    StructField("magType", StringType(), True),
    StructField("type", StringType(), True),
    StructField("title", StringType(), True)
])

"""df_parsed = df.withColumn(
    "geometry",
    F.from_json(F.col("geometry"), geometry_schema)
).withColumn(
    "properties", 
    F.from_json(F.col("properties"), properties_schema)
) """

df = spark.read.table("DE_Learning_LH.dbo.earth_quake").select(
    "id",
    F.from_json(F.col('geometry'), geometry_schema).alias('geo'),
    F.from_json(F.col("properties"), schema=properties_schema).alias('prp')
).select([
    F.col("id"), 
    F.col("geo.type"), 
    F.col("geo.coordinates")[0].alias('longtiude'), 
    F.col("geo.coordinates")[1].alias('latitude'),
    F.col("prp.place"), 
    F.col("prp.title"),
    F.col('prp.sig').alias('sig'),
    F.col('prp.mag').alias('mag'),
    F.col('prp.magType').alias('magType'),
    F.col('prp.time').alias('time'),
    F.col('prp.updated').alias('updated')
])

display(df.limit(5))

StatementMeta(, 54e69200-58a3-4797-94fd-192eee74d181, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fd5586d5-337f-49f0-97f0-b6fcc251d69f)

In [21]:
%%sql
-- Do this at the begining and freeze it after wards 
DROP TABLE DE_Learning_LH.dbo.earth_quake_silver

StatementMeta(, 54e69200-58a3-4797-94fd-192eee74d181, 23, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [22]:
%%sql
-- Do this at the begining and freeze it after wards 
 
CREATE OR REPLACE TABLE DE_Learning_LH.dbo.earth_quake_silver(
    id STRING,
    type STRING,
    longtiude LONG,
    latitude LONG,
    place STRING,
    title STRING,
    sig INT,
    mag LONG,
    magType STRING,
    time TIMESTAMP,
    updated TIMESTAMP
)

StatementMeta(, 54e69200-58a3-4797-94fd-192eee74d181, 24, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [35]:

# ============================================================
# Function: upsert_data
# Purpose:
#   Performs incremental upsert (merge) from streaming micro-batch
#   into the Silver Delta table.
#
# Parameters:
#   microBatchDF : DataFrame → incoming micro-batch from stream
#   batch        : int       → batch identifier (automatically passed)
#
# Notes:
#   - Uses MERGE INTO for id-based deduplication
#   - Inserts only new records (no updates currently)
#   - Assumes id is the unique key
# ============================================================

def upsert_data(microBatchDF, batch):
    # Create a temporary view from the micro-batch
    microBatchDF.createOrReplaceTempView('earthquake_microbatch')

    # MERGE statement (Insert-only pattern)
    sql_query = """
        MERGE INTO DE_Learning_LH.dbo.earth_quake_silver o
        USING earthquake_microbatch b
        ON o.id = b.id
        WHEN NOT MATCHED THEN INSERT *
    """

    
    # Execute SQL (preferred for modern runtimes)
    microBatchDF.sparkSession.sql(sql_query)

    # Alternative (older runtime support):
    # microBatchDF._jdf.sparkSession().sql(sql_query)



# ============================================================
# Function: process_earth_quake_silver
# Purpose:
#   Reads Bronze streaming table, parses JSON columns,
#   applies deduplication using watermarking, and writes
#   to Silver using upsert (MERGE INTO).
#
# Pipeline:
#   Bronze (raw JSON strings)
#        → parse geometry & properties
#        → flatten & transform
#        → watermark for late data handling
#        → deduplicate
#        → upsert into Silver
# ============================================================
def process_earth_quake_silver():
    
    # --------------------------------------------------------
    # Step 1: Read streaming data from Bronze Delta table
    # --------------------------------------------------------
    dedup_stream = (
        spark.readStream.table('DE_Learning_LH.dbo.earth_quake_bronze')
            
            # ------------------------------------------------
            # Step 2: Parse JSON columns into structured format
            # ------------------------------------------------
            .select(
                "id",
                
                # Convert geometry JSON string → struct
                F.from_json(F.col('geometry'), geometry_schema).alias('geo'),
                
                # Convert properties JSON string → struct
                F.from_json(F.col("properties"), properties_schema).alias('prp')

            )
            
            # ------------------------------------------------
            # Step 3: Flatten nested fields
            # ------------------------------------------------
            .select([
                F.col("id"), 

                #Geometry fields
                F.col("geo.type"), 
                F.col("geo.coordinates")[0].alias('longtiude'), 
                F.col("geo.coordinates")[1].alias('latitude'),

                # Properties fields
                F.col("prp.place"), 
                F.col("prp.title"),
                F.col('prp.sig').alias('sig'),
                F.col('prp.mag').alias('mag'),
                F.col('prp.magType').alias('magType'),
                
                # ------------------------------------------------
                # Convert epoch milliseconds → timestamp
                # Required for watermark & time-based operations
                # ------------------------------------------------
                (F.col('prp.time') / 1000).cast("timestamp").alias('time'),
                (F.col('prp.updated') / 1000).alias('updated')
            ])
            
            # ------------------------------------------------
            # Step 4: Watermark (handles late arriving data)
            # Keeps state for 30 seconds
            # -----------------------------------------------
            .withWatermark('time', "30 seconds")
            
            # ------------------------------------------------
            # Step 5: Deduplication
            # Removes duplicate events based on id + event_time
            # ------------------------------------------------
            .dropDuplicates(['id', 'time'])
    )

    
    # --------------------------------------------------------
    # Step 6: Write stream using foreachBatch (upsert pattern)
    # --------------------------------------------------------
    df = (
        dedup_stream
            .writeStream

            # Use foreachBatch for custom MERGE logi
            .foreachBatch(upsert_data)
            
            # ------------------------------------------------
            # Checkpointing (CRITICAL for fault tolerance)
            # ------------------------------------------------
            .option("checkpointLocation", "Files/Bronze/checkpoints/silver_earth_quake/")
            
            # ------------------------------------------------
            # Trigger:
            # availableNow = batch-style streaming (Fabric best practice)
            # ------------------------------------------------
            .trigger(availableNow=True)
            .start()
    )
    # Wait for completion
    df.awaitTermination()


process_earth_quake_silver()

StatementMeta(, 54e69200-58a3-4797-94fd-192eee74d181, 37, Finished, Available, Finished, False)

In [36]:
df = spark.sql("SELECT * FROM DE_Learning_LH.dbo.earthquake_events_silver LIMIT 1000")
display(df.limit(5))

StatementMeta(, 54e69200-58a3-4797-94fd-192eee74d181, 38, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0be72db6-c5b5-4520-9e32-8026ccec9883)